In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/vishwashn/preprocessing/processed_datasets.zip
/kaggle/input/notebooks/vishwashn/preprocessing/__results__.html
/kaggle/input/notebooks/vishwashn/preprocessing/__notebook__.ipynb
/kaggle/input/notebooks/vishwashn/preprocessing/__output__.json
/kaggle/input/notebooks/vishwashn/preprocessing/custom.css
/kaggle/input/notebooks/vishwashn/preprocessing/processed/ticket_rag_docs.json
/kaggle/input/notebooks/vishwashn/preprocessing/processed/amazon_rag_docs.json
/kaggle/input/notebooks/vishwashn/preprocessing/processed/olist_rag_docs.json
/kaggle/input/notebooks/vishwashn/preprocessing/processed/master_df.parquet
/kaggle/input/notebooks/vishwashn/preprocessing/processed/amazon_complaints.parquet
/kaggle/input/notebooks/vishwashn/preprocessing/processed/unified_rag_corpus.json
/kaggle/input/notebooks/vishwashn/preprocessing/processed/policy_doc.txt
/kaggle/input/notebooks/vishwashn/preprocessing/processed/synthetic_tickets.parquet
/kaggle/input/notebooks/vishwashn/prepr

# ================= PHASE 2 ================= 

# SECTION 1 - NOTEBOOK SETUP

In [2]:
import numpy as np
print(np.__version__)

1.26.4


## Install all Phase 2 dependencies

## Load Corpus from Phase 1

In [3]:
# CELL 1 — MINIMAL INSTALL: only what is genuinely missing from Kaggle
# FAISS, numpy, torch, pandas are already on Kaggle — do NOT reinstall them
# Only install sentence-transformers and langchain which Kaggle does not ship

# Step 1 — Fix numpy ONLY if you see numpy errors (uncomment, run, restart, re-comment)
# !pip install -q --force-reinstall numpy==1.26.4

# Step 2 — Install only the missing libraries
!pip install -q faiss-cpu
!pip install -q sentence-transformers==2.7.0
!pip install -q langchain==0.1.20 langchain-community==0.0.38
!pip install -q tqdm

# That is all. No chromadb. No hnswlib. No version conflicts.

# Step 3 — Verify
import numpy as np
import torch
import faiss
import sentence_transformers

print(f"NumPy:                {np.__version__}")
print(f"PyTorch:              {torch.__version__}")
print(f"FAISS version:        {faiss.__version__}")
print(f"SentenceTransformers: {sentence_transformers.__version__}")
print(f"GPU available:        {torch.cuda.is_available()}")
print(f"GPU name:             {torch.cuda.get_device_name(0) if torch.cuda.is_available() else chr(67)+chr(80)+chr(85)}")
print()
print("✓ All dependencies verified — RESTART SESSION NOW, then run from Cell 2")


NumPy:                1.26.4
PyTorch:              2.10.0+cu128
FAISS version:        1.13.2
SentenceTransformers: 2.7.0
GPU available:        True
GPU name:             Tesla T4

✓ All dependencies verified — RESTART SESSION NOW, then run from Cell 2


In [7]:
# CELL 2 — Load unified RAG corpus from Phase 1 output
import json, os, pandas as pd
from tqdm import tqdm
from collections import Counter

# Adjust this path to your Phase 1 notebook name
PHASE1_PATH = '/kaggle/input/notebooks/vishwashn/preprocessing/processed/'
# If Phase 1 ran in the same session use:
# PHASE1_PATH = '/kaggle/working/processed/'

with open(PHASE1_PATH + 'unified_rag_corpus.json', 'r', encoding='utf-8') as f:
    corpus = json.load(f)

order_lookup_df = pd.read_parquet(PHASE1_PATH + 'order_lookup.parquet')

print(f"Total documents in corpus: {len(corpus):,}")
print("\nSource distribution:")
sources = Counter(d['source'] for d in corpus)
for src, cnt in sorted(sources.items(), key=lambda x: -x[1]):
    print(f"  {src:30s}: {cnt:,}")


Total documents in corpus: 88,151

Source distribution:
  olist_review                  : 37,370
  amazon_complaint              : 29,986
  amazon_positive               : 10,000
  support_ticket                : 7,699
  seller_kpi                    : 3,095
  policy                        : 1


# SECTION 2 - SMART CHUNKING STRATEGY

## Source-Aware Chunker

In [8]:
# CELL 3 — Source-aware chunking with correct separator order
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Separator order: paragraph breaks BEFORE sentence ends
# "\n\n" and "\n" must come before ". " to avoid splitting on abbreviations
CHUNK_CONFIG = {
    'olist_review':     {'chunk_size': 300, 'chunk_overlap': 50,
                         'separators': ['\n\n', '\n', '. ', '? ', '! ', ', ', ' ', '']},
    'amazon_complaint': {'chunk_size': 400, 'chunk_overlap': 80,
                         'separators': ['\n\n', '\n', '. ', '? ', '! ', ', ', ' ', '']},
    'amazon_positive':  {'chunk_size': 400, 'chunk_overlap': 80,
                         'separators': ['\n\n', '\n', '. ', '? ', '! ', ', ', ' ', '']},
    'support_ticket':   {'chunk_size': 350, 'chunk_overlap': 60,
                         'separators': ['\n\n', '\n', '. ', '! ', ', ', ' ', '']},
    'seller_kpi':       {'chunk_size': 500, 'chunk_overlap': 0,
                         'separators': ['\n', ' ', '']},
    # Policy: section dividers (=====) are highest priority split points
    'policy':           {'chunk_size': 500, 'chunk_overlap': 100,
                         'separators': ['\n=====\n', '\n\n', '\n', '. ', ' ', '']},
}
DEFAULT_CONFIG = {'chunk_size': 400, 'chunk_overlap': 60,
                  'separators': ['\n\n', '\n', '. ', ', ', ' ', '']}

def make_splitter(source):
    cfg = CHUNK_CONFIG.get(source, DEFAULT_CONFIG)
    return RecursiveCharacterTextSplitter(
        chunk_size=cfg['chunk_size'],
        chunk_overlap=cfg['chunk_overlap'],
        separators=cfg['separators'],
        length_function=len
    )

splitters = {src: make_splitter(src) for src in CHUNK_CONFIG}
splitters['default'] = make_splitter('default')

print("✓ Chunkers ready")
for src, cfg in CHUNK_CONFIG.items():
    print(f"  {src:20s}: size={cfg['chunk_size']}, overlap={cfg['chunk_overlap']}")


✓ Chunkers ready
  olist_review        : size=300, overlap=50
  amazon_complaint    : size=400, overlap=80
  amazon_positive     : size=400, overlap=80
  support_ticket      : size=350, overlap=60
  seller_kpi          : size=500, overlap=0
  policy              : size=500, overlap=100


## Chunk All Documents — With Dedup and Size Filter

In [9]:
# CELL 4 — Chunking with deduplication, size filter, and metadata fix
import hashlib

chunked_docs = []
chunk_stats  = Counter()
seen_hashes  = set()

CHUNK_MIN = 20   # drop micro-chunks
CHUNK_MAX = 650  # drop hard-split oversized chunks

for doc in tqdm(corpus, desc='Chunking'):
    source = doc.get('source', 'unknown')
    text   = doc.get('text', '')
    meta   = doc.get('metadata', {})
    doc_id = doc.get('doc_id', '')

    if not text or len(text.strip()) < 10:
        continue

    splitter = splitters.get(source, splitters['default'])
    chunks   = splitter.split_text(text)

    for i, chunk in enumerate(chunks):
        chunk = chunk.strip()
        if len(chunk) < CHUNK_MIN or len(chunk) > CHUNK_MAX:
            continue
        h = hashlib.md5(chunk.lower().encode()).hexdigest()
        if h in seen_hashes:
            continue
        seen_hashes.add(h)
        # metadata: **meta first so explicit keys below always win
        chunk_meta = {**meta, 'source': source, 'doc_id': doc_id, 'chunk_idx': i}
        chunked_docs.append({
            'chunk_id': f'{doc_id}_c{i}',
            'text':     chunk,
            'source':   source,
            'doc_id':   doc_id,
            'chunk_idx':i,
            'metadata': chunk_meta
        })
        chunk_stats[source] += 1

print(f"\n✓ Total chunks: {len(chunked_docs):,}")
for src, cnt in sorted(chunk_stats.items(), key=lambda x: -x[1]):
    print(f"  {src:30s}: {cnt:,}")


Chunking: 100%|██████████| 88151/88151 [00:01<00:00, 45226.82it/s]


✓ Total chunks: 130,913
  amazon_complaint              : 61,985
  olist_review                  : 41,742
  amazon_positive               : 14,539
  support_ticket                : 9,541
  seller_kpi                    : 3,095
  policy                        : 11


## Chunk quality inspection

In [10]:
# CELL 5 — Inspect chunk quality before embedding
import random, numpy as np

lengths = [len(c['text']) for c in chunked_docs]
print("=== CHUNK STATISTICS ===")
print(f"  Min:    {min(lengths):,} chars")
print(f"  Max:    {max(lengths):,} chars")
print(f"  Mean:   {np.mean(lengths):.0f} chars")
print(f"  Median: {np.median(lengths):.0f} chars")
print(f"  <20 chars (noise):    {sum(1 for l in lengths if l < 20):,}  ← should be 0")
print(f"  >650 chars (too long): {sum(1 for l in lengths if l > 650):,}  ← should be 0")

for src in ['olist_review', 'support_ticket', 'policy']:
    sample = [c for c in chunked_docs if c['source'] == src]
    if sample:
        c = random.choice(sample[:200])
        print(f"\n--- {src} ---")
        print(c['text'][:250])
        print(f"length={len(c['text'])} | chunk_id={c['chunk_id']}")



=== CHUNK STATISTICS ===
  Min:    20 chars
  Max:    494 chars
  Mean:   249 chars
  Median: 249 chars
  <20 chars (noise):    0  ← should be 0
  >650 chars (too long): 0  ← should be 0

--- olist_review ---
Order ID: 95e42e6aaf6264cd3e77c06b32dc3003. Category: sports_leisure. Review score: 1/5. Delivery status: on time. Customer feedback: They did not deliver the product and have not been contacted to resolve the problem so far.
length=225 | chunk_id=review_95e42e6aaf6264cd3e77c06b32dc3003_c0

--- support_ticket ---
Support ticket (delivery delay). Priority: medium. Channel: app. Customer query: Tracking shows my order 20e64aa65fbb0149e274381104f46bbe left the warehouse but nothing has arrived. Estimated date was December 05, 2017. Resolved by: We have contacted
length=346 | chunk_id=TKT-F06E8BB1_c0

--- policy ---
3.1 DISPATCH SLA
Sellers must dispatch orders within 3 business days of order confirmation.
Failure to dispatch results in automatic order cancellation and full refund.

3

# SECTION 3 — GENERATING EMBEDDINGS

# Load Embedding Model

In [11]:
# CELL 6 — Load sentence-transformers model
from sentence_transformers import SentenceTransformer
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
embed_model = SentenceTransformer(MODEL_NAME, device=DEVICE)

test_vec = embed_model.encode('test')
print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Embedding dimensions: {len(test_vec)} (expected 384)")


Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Model loaded: sentence-transformers/all-MiniLM-L6-v2
  Embedding dimensions: 384 (expected 384)


## Generate Embeddings — Batch GPU Encode

In [12]:
# CELL 7 — Batch encode all chunks on GPU
import numpy as np
from tqdm import tqdm

BATCH_SIZE = 256
texts = [c['text'] for c in chunked_docs]

print(f"Embedding {len(texts):,} chunks...")
print(f"Estimated time: ~{max(2, len(texts)//10000)} minutes on GPU")

embeddings = embed_model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2 normalise — cosine sim = dot product
    convert_to_numpy=True
)

print(f'\n✓ Embeddings generated')
print(f"\n✓ Embeddings shape: {embeddings.shape}")   # (N, 384)
print(f"  Dtype:  {embeddings.dtype}")
print(f"  Memory: {embeddings.nbytes/1024**2:.1f} MB")

# Verify normalisation
norms = np.linalg.norm(embeddings, axis=1)
print(f"  All L2-normalised: {np.allclose(norms, 1.0, atol=1e-5)}")  # must be True


Embedding 130,913 chunks...
Estimated time: ~13 minutes on GPU


Batches:   0%|          | 0/512 [00:00<?, ?it/s]


✓ Embeddings generated

✓ Embeddings shape: (130913, 384)
  Dtype:  float32
  Memory: 191.8 MB
  All L2-normalised: True


## Save Checkpoint — Embeddings + Chunks to Disk

In [13]:
# CELL 8 — Save embeddings and chunks to disk (primary checkpoint)
import os, json

os.makedirs('/kaggle/working/embeddings/', exist_ok=True)

# Save embedding matrix
np.save('/kaggle/working/embeddings/vectors.npy', embeddings)

# Save full chunks (text + metadata) for index rebuild
with open('/kaggle/working/embeddings/chunks_full.json', 'w', encoding='utf-8') as f:
    json.dump(chunked_docs, f, ensure_ascii=False)

# Save lightweight metadata only (for quick inspection)
chunk_meta_list = [{
    'chunk_id':  c['chunk_id'],
    'doc_id':    c['doc_id'],
    'source':    c['source'],
    'chunk_idx': c['chunk_idx'],
    'text_len':  len(c['text']),
} for c in chunked_docs]

with open('/kaggle/working/embeddings/chunk_meta.json', 'w') as f:
    json.dump(chunk_meta_list, f)

size = os.path.getsize('/kaggle/working/embeddings/vectors.npy')/1024**2
print(f"✓ Checkpoint saved:")
print(f"  vectors.npy:      {size:.1f} MB")
print(f"  chunks_full.json: {len(chunked_docs):,} records")


✓ Checkpoint saved:
  vectors.npy:      191.8 MB
  chunks_full.json: 130,913 records


# SECTION 4 — BUILDING THE FAISS VECTOR DATABASE

## ChromaDB Setup — EphemeralClient

In [14]:
# CELL 9 — BUILD FAISS INDEX
# faiss is pre-installed on Kaggle — no install needed
import faiss
import numpy as np

DIM = embeddings.shape[1]   # 384
N   = embeddings.shape[0]

print(f"Building FAISS index: {N:,} vectors, {DIM} dimensions")

# IndexFlatIP = exact Inner Product search
# On L2-normalised vectors, inner product == cosine similarity
# For < 500K vectors, exact search is fast enough (< 5ms per query on CPU)
# Use IndexHNSWFlat for > 500K vectors (approximate but faster)

if N < 500_000:
    # Exact search — best recall, simple
    index = faiss.IndexFlatIP(DIM)
    print("  Using: IndexFlatIP (exact cosine — best for < 500K vectors)")
else:
    # Approximate HNSW search — fast even at millions of vectors
    index = faiss.IndexHNSWFlat(DIM, 32)   # 32 = M parameter
    index.hnsw.efConstruction = 200
    index.hnsw.efSearch = 100
    print("  Using: IndexHNSWFlat (approximate — M=32, ef=200)")

# Add all vectors
emb_float32 = embeddings.astype('float32')   # FAISS requires float32
index.add(emb_float32)

print(f"\n✓ FAISS index built")
print(f"  Total vectors indexed: {index.ntotal:,}")

# Quick search test
test_q = embed_model.encode(['return policy damaged product'],
                             normalize_embeddings=True).astype('float32')
scores, indices = index.search(test_q, 3)
print(f"  Test query top scores: {scores[0].tolist()}")
print(f"  Test query top chunk:  {chunked_docs[indices[0][0]]['text'][:100]}...")


Building FAISS index: 130,913 vectors, 384 dimensions
  Using: IndexFlatIP (exact cosine — best for < 500K vectors)

✓ FAISS index built
  Total vectors indexed: 130,913
  Test query top scores: [0.6336708068847656, 0.586524486541748, 0.576866626739502]
  Test query top chunk:  . But I didn't buy this item to test the return policy....


## Build Source-Specific Sub-Indexes

In [15]:
# CELL 10 — Build three focused FAISS indexes (replaces 3 ChromaDB collections)

# Helper: build index from a subset of chunks
def build_sub_index(source_filter):
    """Build a FAISS index for chunks matching source_filter (str or set)."""
    if isinstance(source_filter, str):
        source_filter = {source_filter}
    indices = [i for i, c in enumerate(chunked_docs) if c['source'] in source_filter]
    if not indices:
        print(f"  WARNING: no chunks found for {source_filter}")
        return None, [], []
    sub_emb = emb_float32[indices]
    sub_idx = faiss.IndexFlatIP(DIM)
    sub_idx.add(sub_emb)
    sub_chunks = [chunked_docs[i] for i in indices]
    print(f"  {str(source_filter):45s}: {sub_idx.ntotal:,} vectors")
    return sub_idx, sub_chunks, indices

# Build all three
print("Building sub-indexes...")

# 1. Policy index — for refund/return/policy queries
idx_policy,  chunks_policy,  _ = build_sub_index('policy')

# 2. Complaints index — for broken/defective/late/lost queries
COMPLAINT_SRCS = {'olist_review', 'amazon_complaint', 'support_ticket'}
idx_complaints, chunks_complaints, _ = build_sub_index(COMPLAINT_SRCS)

# 3. Main index already built in Cell 9 (all sources)
idx_main   = index
chunks_main = chunked_docs

print(f"\n✓ All indexes ready:")
print(f"  main:       {idx_main.ntotal:,}")
print(f"  policy:     {idx_policy.ntotal:,}")
print(f"  complaints: {idx_complaints.ntotal:,}")


Building sub-indexes...
  {'policy'}                                   : 11 vectors
  {'amazon_complaint', 'support_ticket', 'olist_review'}: 113,268 vectors

✓ All indexes ready:
  main:       130,913
  policy:     11
  complaints: 113,268


## Save FAISS Indexes to Disk — Persistence

In [16]:
# CELL 11 — Persist FAISS indexes
# On session restart: load indexes + chunks_full.json (Cell 11R below)
import os

FAISS_DIR = '/kaggle/working/faiss_indexes/'
os.makedirs(FAISS_DIR, exist_ok=True)

faiss.write_index(idx_main,       FAISS_DIR + 'main.index')
faiss.write_index(idx_policy,     FAISS_DIR + 'policy.index')
faiss.write_index(idx_complaints, FAISS_DIR + 'complaints.index')

# Save chunk lists aligned to each index
with open(FAISS_DIR + 'chunks_main.json',       'w') as f: json.dump(chunks_main, f)
with open(FAISS_DIR + 'chunks_policy.json',     'w') as f: json.dump(chunks_policy, f)
with open(FAISS_DIR + 'chunks_complaints.json', 'w') as f: json.dump(chunks_complaints, f)

for fname in os.listdir(FAISS_DIR):
    sz = os.path.getsize(FAISS_DIR+fname)/1024**2
    print(f"  {fname:35s}: {sz:.1f} MB")
print("\n✓ All indexes saved to disk")


  chunks_complaints.json             : 74.0 MB
  policy.index                       : 0.0 MB
  chunks_policy.json                 : 0.0 MB
  chunks_main.json                   : 82.7 MB
  complaints.index                   : 165.9 MB
  main.index                         : 191.8 MB

✓ All indexes saved to disk


## SESSION RESTORE — Reload Indexes After Restart (Run Instead of Cells 6–11)
**NOTE:**	After a session restart, run Cell 1 (deps only), Cell 2 (corpus), Cells 3–5 (chunking), then jump here. Skips re-embedding entirely. Takes about 20 seconds.

In [ ]:
# CELL 11R — RESTORE: load saved FAISS indexes + embeddings (no re-embedding)
import faiss, json, numpy as np

FAISS_DIR = '/kaggle/working/faiss_indexes/'

# Reload embedding model (needed for query embedding)
from sentence_transformers import SentenceTransformer
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
print(f"✓ Embed model loaded on {DEVICE}")

# Load indexes
idx_main       = faiss.read_index(FAISS_DIR + 'main.index')
idx_policy     = faiss.read_index(FAISS_DIR + 'policy.index')
idx_complaints = faiss.read_index(FAISS_DIR + 'complaints.index')

# Load aligned chunk lists
with open(FAISS_DIR + 'chunks_main.json')       as f: chunks_main       = json.load(f)
with open(FAISS_DIR + 'chunks_policy.json')     as f: chunks_policy     = json.load(f)
with open(FAISS_DIR + 'chunks_complaints.json') as f: chunks_complaints = json.load(f)

DIM = 384

print(f"✓ Indexes restored:")
print(f"  main:       {idx_main.ntotal:,}")
print(f"  policy:     {idx_policy.ntotal:,}")
print(f"  complaints: {idx_complaints.ntotal:,}")
print("\nReady — continue from Cell 12 (retriever)")


# SECTION 5 — RETRIEVAL MODULE & QUERY ROUTING

## FAISSRetriever Class — Full Implementation

In [18]:
# CELL 12 — FAISSRetriever: drop-in replacement for OlistRetriever
import numpy as np
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class RetrievedDoc:
    text:       str
    source:     str
    metadata:   dict
    similarity: float = 0.0


class FAISSRetriever:
    """
    FAISS-based retriever with intent routing across three indexes.
    Interface identical to the original OlistRetriever.
    Works with Phase 3 RAG chain and LangGraph agent without any changes.
    """

    REFUND_KW  = ['refund','return','cancel','money back','chargeback']
    POLICY_KW  = ['policy','rules','allowed','eligible','rights','how many days','cdc']
    COMPLAINT_KW = ['broken','defective','late','delayed','lost','wrong item','damaged','not working']

    def __init__(self, embed_model, idx_main, chunks_main,
                 idx_policy, chunks_policy,
                 idx_complaints, chunks_complaints):
        self.model      = embed_model
        self.indexes    = {
            'main':       (idx_main,       chunks_main),
            'policy':     (idx_policy,     chunks_policy),
            'complaints': (idx_complaints, chunks_complaints),
        }

    def _route(self, query: str) -> str:
        q = query.lower()
        if any(k in q for k in self.REFUND_KW + self.POLICY_KW):
            return 'policy'
        if any(k in q for k in self.COMPLAINT_KW):
            return 'complaints'
        return 'main'

    def _embed_query(self, query: str) -> np.ndarray:
        return self.model.encode([query],
                                 normalize_embeddings=True
                                 ).astype('float32')

    def retrieve(
        self,
        query:         str,
        k:             int   = 5,
        min_score:     float = 0.25,
        source_filter: Optional[str] = None,
    ) -> list:
        """
        Retrieve top-k documents for query.
        Returns list of dicts with keys: text, source, metadata, similarity
        """
        route = source_filter if source_filter in self.indexes else self._route(query)
        index, chunks = self.indexes[route]

        q_vec = self._embed_query(query)

        # Search — fetch 2x to allow score filtering
        fetch_k = min(k * 2, index.ntotal)
        scores, indices = index.search(q_vec, fetch_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:          # FAISS returns -1 for unfilled slots
                continue
            sim = float(score)     # inner product on normalised vecs = cosine sim
            if sim < min_score:
                continue
            c = chunks[idx]
            results.append({
                'text':       c['text'],
                'source':     c['source'],
                'metadata':   c['metadata'],
                'similarity': round(sim, 4),
            })
            if len(results) >= k:
                break

        return results


# ── Instantiate ─────────────────────────────────────────────
retriever = FAISSRetriever(
    embed_model    = embed_model,
    idx_main       = idx_main,       chunks_main       = chunks_main,
    idx_policy     = idx_policy,     chunks_policy     = chunks_policy,
    idx_complaints = idx_complaints, chunks_complaints = chunks_complaints,
)
print("✓ FAISSRetriever ready")

# Quick test
test = retriever.retrieve('damaged product want refund', k=3)
for r in test:
    print(f"  [{r['source']}] sim={r['similarity']:.3f}  {r['text'][:80]}...")



✓ FAISSRetriever ready
  [policy] sim=0.529  2.4 WRONG ITEM DELIVERED
If you received the wrong item:
1. Take photos of the r...
  [policy] sim=0.466  1.2 EXTENDED RETURN FOR DEFECTIVE PRODUCTS
For defective or damaged products, th...
  [policy] sim=0.451  1.4 REFUND ELIGIBILITY CONDITIONS
Full refund is issued when:
- Product is retur...


## Test Retrieval — 5 Smoke Tests

In [19]:
# CELL 13 — 5 smoke tests covering all query intent types
SMOKE_TESTS = [
    {'q': 'What is the return policy for damaged products?',
     'expected_route': 'policy',
     'desc': 'Policy query'},
    {'q': 'My order is delayed by 10 days what should I do',
     'expected_route': 'complaints',
     'desc': 'Delay complaint'},
    {'q': 'The electronics I received is broken and not working',
     'expected_route': 'complaints',
     'desc': 'Defective product'},
    {'q': 'How long does a refund take to process',
     'expected_route': 'policy',
     'desc': 'Refund timeline'},
    {'q': 'Seller is not responding and my package never arrived',
     'expected_route': 'complaints',
     'desc': 'Lost package + seller'},
]

print("=== SMOKE TESTS ===")
for t in SMOKE_TESTS:
    route  = retriever._route(t['q'])
    results = retriever.retrieve(t['q'], k=3)
    ok = "✓" if route == t['expected_route'] else "✗"
    print(f"\n{ok} [{t['desc']}]")
    print(f"  Routed to: {route} (expected: {t['expected_route']})")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] sim={r['similarity']:.3f} src={r['source']}  {r['text'][:100]}...")


=== SMOKE TESTS ===

✓ [Policy query]
  Routed to: policy (expected: policy)
  [1] sim=0.601 src=policy  1.2 EXTENDED RETURN FOR DEFECTIVE PRODUCTS
For defective or damaged products, the return window is e...
  [2] sim=0.559 src=policy  1.1 RETURN WINDOW
Customers have the right to return any product within 7 (seven) calendar days
of r...
  [3] sim=0.548 src=policy  1.4 REFUND ELIGIBILITY CONDITIONS
Full refund is issued when:
- Product is returned within the 7-day...

✓ [Delay complaint]
  Routed to: complaints (expected: complaints)
  [1] sim=0.758 src=support_ticket  . Order ID: ed9b3b9e44dcd0fdbe1bffbdeaf4d2ce. Please help. Resolved by: We have contacted the logist...
  [2] sim=0.755 src=support_ticket  . Order ID: 10e652c1735375eb89f51e95212615e8. Please help. Resolved by: We have contacted the logist...
  [3] sim=0.751 src=support_ticket  . Order ID: a8e0b9e3c94ebc0f5d0ebbfa4d6110b7. Please help. Resolved by: We have contacted the logist...

✓ [Defective product]
  Routed to: com

## Precision@5 Retrieval Quality Measurement

In [20]:
# CELL 14 — Precision@5 using keyword proxy labels
RELEVANCE_TESTS = [
    {'query': 'refund policy 7 days return window',
     'keywords': ['7 day','return','refund','CDC','policy'],
     'expected_src': 'policy'},
    {'query': 'product broken defective stopped working',
     'keywords': ['broken','defective','stopped','damaged','dead'],
     'expected_src': 'amazon_complaint'},
    {'query': 'delivery delayed package not arrived late',
     'keywords': ['delay','late','not arrived','overdue','estimated'],
     'expected_src': 'olist_review'},
    {'query': 'seller not shipping order complaint',
     'keywords': ['seller','dispatch','shipping','complaint','not sent'],
     'expected_src': 'support_ticket'},
]

results_summary = []
for t in RELEVANCE_TESTS:
    hits = retriever.retrieve(t['query'], k=5)
    precision = sum(
        1 for h in hits
        if any(kw.lower() in h['text'].lower() for kw in t['keywords'])
    ) / max(len(hits), 1)
    src_hits = sum(1 for h in hits if h['source'] == t['expected_src'])
    results_summary.append({
        'query':       t['query'][:40],
        'precision@5': round(precision, 2),
        'source_hits': f'{src_hits}/5'
    })

import pandas as pd
df = pd.DataFrame(results_summary)
print(df.to_string(index=False))
avg_p = sum(r['precision@5'] for r in results_summary) / len(results_summary)
print(f"\nAverage Precision@5: {avg_p:.2f}  (target > 0.60)")


                                   query  precision@5 source_hits
      refund policy 7 days return window          1.0         5/5
product broken defective stopped working          1.0         0/5
delivery delayed package not arrived lat          1.0         2/5
     seller not shipping order complaint          1.0         0/5

Average Precision@5: 1.00  (target > 0.60)


# SECTION 6 — LANGCHAIN WRAPPER

## LangChain-Compatible Retriever Wrapper 

In [22]:
# CELL 15 — LangChain wrapper around FAISSRetriever
# Plug this into Phase 3 RetrievalQA chain and LangGraph agent tools
from langchain.schema import BaseRetriever, Document
from langchain.callbacks.manager import CallbackManagerForRetrieverRun
from typing import List


class LangChainFAISSRetriever(BaseRetriever):
    """LangChain-compatible wrapper. Identical interface to original guide."""
    retriever:  object
    k:          int   = 5
    min_score:  float = 0.25

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        raw = self.retriever.retrieve(query, k=self.k, min_score=self.min_score)
        return [
            Document(
                page_content=r['text'],
                metadata={**r['metadata'], 'similarity': r['similarity'],
                          'source': r['source']}
            )
            for r in raw
        ]


lc_retriever = LangChainFAISSRetriever(retriever=retriever, k=5, min_score=0.25)

# Test it
docs = lc_retriever.get_relevant_documents('product not working want refund')
print(f"✓ LangChain wrapper works: {len(docs)} docs returned")
for d in docs:
    print(f"  [{d.metadata['source']}] sim={d.metadata['similarity']:.3f}  {d.page_content[:90]}...")


✓ LangChain wrapper works: 5 docs returned
  [policy] sim=0.439  2.4 WRONG ITEM DELIVERED
If you received the wrong item:
1. Take photos of the received it...
  [policy] sim=0.418  1.4 REFUND ELIGIBILITY CONDITIONS
Full refund is issued when:
- Product is returned within...
  [policy] sim=0.396  1.2 EXTENDED RETURN FOR DEFECTIVE PRODUCTS
For defective or damaged products, the return w...
  [policy] sim=0.372  1.1 RETURN WINDOW
Customers have the right to return any product within 7 (seven) calendar...
  [policy] sim=0.361  4.1 Under Brazilian law, customers have the right to:
- Clear and accurate product informa...


## Save Config for Phase 3

In [23]:
# CELL 16 — Save retriever config for Phase 3 to reload
import json

retriever_config = {
    'vector_store':  'faiss',
    'model_name':    'sentence-transformers/all-MiniLM-L6-v2',
    'faiss_dir':     '/kaggle/working/faiss_indexes/',
    'indexes': {
        'main':       'main.index',
        'policy':     'policy.index',
        'complaints': 'complaints.index',
    },
    'dim': 384,
    'retrieval': {
        'default_k':  5,
        'min_score':  0.25,
        'similarity': 'cosine (inner product on L2-normalised vectors)',
    },
    'chunk_config': {
        src: {'chunk_size': cfg['chunk_size'], 'chunk_overlap': cfg['chunk_overlap']}
        for src, cfg in CHUNK_CONFIG.items()
    },
    'total_chunks': len(chunked_docs),
}

with open('/kaggle/working/retriever_config.json', 'w') as f:
    json.dump(retriever_config, f, indent=2)

print("✓ Config saved to /kaggle/working/retriever_config.json")


✓ Config saved to /kaggle/working/retriever_config.json


# SECTION 7 - FULL VERIFICATION (PHASE 2)

In [24]:
# CELL 17 — Final verification: run this last
import os

print("====== PHASE 2 VERIFICATION REPORT ======")

print("\n[1] FAISS Indexes:")
print(f"    main:       {idx_main.ntotal:,} vectors")
print(f"    policy:     {idx_policy.ntotal:,} vectors")
print(f"    complaints: {idx_complaints.ntotal:,} vectors")

print("\n[2] Embedding matrix:")
vecs = np.load('/kaggle/working/embeddings/vectors.npy')
print(f"    Shape: {vecs.shape}   (chunks × 384)")
print(f"    L2-normalised: {np.allclose(np.linalg.norm(vecs, axis=1), 1.0, atol=1e-5)}")

print("\n[3] Retrieval test:")
r = retriever.retrieve('late delivery refund', k=3)
print(f"    Returned {len(r)} docs, top score: {r[0]['similarity']:.3f}")

print("\n[4] LangChain wrapper test:")
lc_docs = lc_retriever.get_relevant_documents('broken product return policy')
print(f"    Returned {len(lc_docs)} LangChain Document objects")

print("\n[5] Output files:")
FILES = [
    '/kaggle/working/faiss_indexes/main.index',
    '/kaggle/working/faiss_indexes/policy.index',
    '/kaggle/working/faiss_indexes/complaints.index',
    '/kaggle/working/embeddings/vectors.npy',
    '/kaggle/working/embeddings/chunks_full.json',
    '/kaggle/working/retriever_config.json',
]
all_ok = True
for fp in FILES:
    ok   = os.path.exists(fp)
    sz   = os.path.getsize(fp)/1024**2 if os.path.isfile(fp) else 0
    mark = "✓" if ok else "✗"
    print(f"    {mark} {fp.split('/')[-1]:35s} {sz:.1f} MB")
    if not ok: all_ok = False

print()
if all_ok:
    print("====== PHASE 2 COMPLETE — READY FOR PHASE 3 ======")
else:
    print("====== SOME FILES MISSING — RE-RUN FAILED CELLS ======")


====== PHASE 2 VERIFICATION REPORT ======

[1] FAISS Indexes:
    main:       130,913 vectors
    policy:     11 vectors
    complaints: 113,268 vectors

[2] Embedding matrix:
    Shape: (130913, 384)   (chunks × 384)
    L2-normalised: True

[3] Retrieval test:
    Returned 3 docs, top score: 0.573

[4] LangChain wrapper test:
    Returned 5 LangChain Document objects

[5] Output files:
    ✓ main.index                          191.8 MB
    ✓ policy.index                        0.0 MB
    ✓ complaints.index                    165.9 MB
    ✓ vectors.npy                         191.8 MB
    ✓ chunks_full.json                    82.6 MB
    ✓ retriever_config.json               0.0 MB

====== PHASE 2 COMPLETE — READY FOR PHASE 3 ======


# ================= PHASE 3 =================

# SECTION 1 — HYBRID RETRIEVAL STRATEGY

## Install Retrieval Dependencies
BM25 + rank_bm25 for sparse search


In [26]:
# PHASE 3 — CELL 1
# Adds only what Phase 2 did NOT install
# sentence-transformers, langchain, langchain-community → already in Phase 2

!pip install -q rank-bm25
!pip install -q langchain-groq
!pip install -q langgraph langsmith
!pip install -q ragas datasets

print("✓ Phase 3 additional packages installed")
print("  rank-bm25    — BM25 sparse retrieval")
print("  langchain-groq — free LLM API")
print("  langgraph    — agent framework")
print("  ragas        — evaluation metrics")


✓ Phase 3 additional packages installed
  rank-bm25    — BM25 sparse retrieval
  langchain-groq — free LLM API
  langgraph    — agent framework
  ragas        — evaluation metrics


## FAISS Restore + FAISSRetriever + BM25 Build — ALL IN ONE CELL

In [27]:
# PHASE 3 — CELL 2 (THE FAISS UPDATE MENTIONED IN PHASE 2 DOC)
# This is the replacement for the original ChromaDB load cell
# Does: restore FAISS + rebuild FAISSRetriever + build BM25

import json, re, faiss, numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from dataclasses import dataclass
from typing import Optional
import torch

FAISS_DIR = '/kaggle/working/faiss_indexes/'

# ── PART A: Restore FAISS (only if session was restarted) ─────────────
# If running in same session as Phase 2, these variables already exist
if 'embed_model' not in dir() or 'idx_main' not in dir():
    print("Session restarted — restoring FAISS indexes...")
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
    idx_main       = faiss.read_index(FAISS_DIR + 'main.index')
    idx_policy     = faiss.read_index(FAISS_DIR + 'policy.index')
    idx_complaints = faiss.read_index(FAISS_DIR + 'complaints.index')
    with open(FAISS_DIR + 'chunks_main.json')       as f: chunks_main       = json.load(f)
    with open(FAISS_DIR + 'chunks_policy.json')     as f: chunks_policy     = json.load(f)
    with open(FAISS_DIR + 'chunks_complaints.json') as f: chunks_complaints = json.load(f)
    DIM = 384
    print("✓ FAISS indexes restored")
else:
    print("✓ FAISS indexes already in memory (same session)")

# ── PART B: Ensure chunked_docs is available ──────────────────────────
if 'chunked_docs' not in dir():
    with open('/kaggle/working/embeddings/chunks_full.json') as f:
        chunked_docs = json.load(f)
    print(f"✓ chunked_docs loaded from disk: {len(chunked_docs):,} chunks")
else:
    print(f"✓ chunked_docs already in memory: {len(chunked_docs):,} chunks")

# ── PART C: FAISSRetriever class definition ───────────────────────────
# This class is used directly AND wrapped inside HybridRetriever (Cell 4)
@dataclass
class RetrievedDoc:
    text:         str
    source:       str
    metadata:     dict
    dense_score:  float = 0.0
    sparse_score: float = 0.0
    rerank_score: float = 0.0
    final_score:  float = 0.0

class FAISSRetriever:
    """FAISS-based retriever — Phase 2 output, reused in Phase 3."""
    REFUND_KW    = ['refund','return','cancel','money back','chargeback']
    POLICY_KW    = ['policy','rules','allowed','eligible','rights','how many days','cdc']
    COMPLAINT_KW = ['broken','defective','late','delayed','lost','wrong item','damaged','not working']

    def __init__(self, embed_model, idx_main, chunks_main,
                 idx_policy, chunks_policy, idx_complaints, chunks_complaints):
        self.model   = embed_model
        self.indexes = {
            'main':       (idx_main,       chunks_main),
            'policy':     (idx_policy,     chunks_policy),
            'complaints': (idx_complaints, chunks_complaints),
        }

    def _route(self, query: str) -> str:
        q = query.lower()
        if any(k in q for k in self.REFUND_KW + self.POLICY_KW): return 'policy'
        if any(k in q for k in self.COMPLAINT_KW):               return 'complaints'
        return 'main'

    def _embed(self, query: str) -> np.ndarray:
        return self.model.encode([query], normalize_embeddings=True).astype('float32')

    def retrieve(self, query: str, k: int = 5, min_score: float = 0.25,
                 source_filter: Optional[str] = None) -> list:
        route = source_filter if source_filter in self.indexes else self._route(query)
        index, chunks = self.indexes[route]
        q_vec = self._embed(query)
        fetch_k = min(k * 2, index.ntotal)
        scores, idxs = index.search(q_vec, fetch_k)
        results = []
        for score, idx in zip(scores[0], idxs[0]):
            if idx == -1: continue
            if float(score) < min_score: continue
            c = chunks[int(idx)]
            results.append(RetrievedDoc(
                text=c['text'], source=c['source'], metadata=c.get('metadata',{}),
                dense_score=round(float(score),4)
            ))
            if len(results) >= k: break
        return results

faiss_retriever = FAISSRetriever(
    embed_model=embed_model,
    idx_main=idx_main,           chunks_main=chunks_main,
    idx_policy=idx_policy,       chunks_policy=chunks_policy,
    idx_complaints=idx_complaints, chunks_complaints=chunks_complaints,
)

# ── PART D: Build BM25 sparse index ───────────────────────────────────
def tokenize_for_bm25(text: str) -> list:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    stopwords = {'the','a','an','is','it','in','on','at','to','for',
                 'of','and','or','but','not','this','that','with','was'}
    return [t for t in text.split() if t not in stopwords and len(t) > 1]

print("Building BM25 index...")
corpus_tokens = [tokenize_for_bm25(c['text']) for c in chunked_docs]
bm25_index    = BM25Okapi(corpus_tokens)

print(f"✓ BM25 index built: {len(corpus_tokens):,} documents")
print("  Cell 2 complete: FAISS + FAISSRetriever + BM25 all ready")


✓ FAISS indexes already in memory (same session)
✓ chunked_docs already in memory: 130,913 chunks
Building BM25 index...
✓ BM25 index built: 130,913 documents
  Cell 2 complete: FAISS + FAISSRetriever + BM25 all ready


## Cross-Encoder Re-ranker

In [29]:
# PHASE 3 — CELL 3 — UNCHANGED
from sentence_transformers import CrossEncoder

RERANKER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
reranker = CrossEncoder(RERANKER_MODEL, max_length=512)

test_score = reranker.predict([('what is return policy', 'You can return within 7 days')])
print(f"✓ Cross-encoder loaded: {RERANKER_MODEL}")
print(f"  Test relevance score: {test_score[0]:.3f}")

✓ Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
  Test relevance score: -6.308


## HybridRetriever — FAISS Dense + BM25 Sparse + RRF + Cross-Encoder

In [31]:
# PHASE 3 — CELL 4 — CHANGED: HybridRetriever (FAISS version)
import numpy as np
from typing import Optional

class HybridRetriever:
    """
    Hybrid retrieval: FAISS dense + BM25 sparse + RRF merge + cross-encoder rerank.
    Identical interface to original guide. ChromaDB replaced with FAISS.
    """
    PRIORITY_BOOST = {'high': 0.15, 'medium': 0.05, 'low': 0.0}

    def __init__(self, faiss_retriever, bm25_index, chunked_docs, reranker,
                 dense_weight=0.6, sparse_weight=0.4):
        self.fr       = faiss_retriever   # FAISSRetriever from Cell 2
        self.bm25     = bm25_index
        self.docs     = chunked_docs
        self.reranker = reranker
        self.dw, self.sw = dense_weight, sparse_weight

    # ── Dense retrieval (FAISS) ───────────────────────────────
    def _dense_retrieve(self, query: str, k: int) -> list:
        return self.fr.retrieve(query, k=k, min_score=0.0)  # no score filter here

    # ── Sparse retrieval (BM25) ───────────────────────────────
    def _sparse_retrieve(self, query: str, k: int) -> list:
        tokens = tokenize_for_bm25(query)
        scores = self.bm25.get_scores(tokens)
        top_k  = np.argsort(scores)[::-1][:k]
        max_s  = scores[top_k[0]] if scores[top_k[0]] > 0 else 1.0
        docs   = []
        for idx in top_k:
            if scores[idx] > 0:
                d = self.docs[int(idx)]
                docs.append(RetrievedDoc(
                    text=d['text'], source=d['source'], metadata=d.get('metadata',{}),
                    sparse_score=round(float(scores[idx])/max_s, 4)
                ))
        return docs

    # ── Reciprocal Rank Fusion ────────────────────────────────
    def _rrf(self, dense, sparse, k_rrf=60) -> list:
        scores, dmap = {}, {}
        for rank, doc in enumerate(dense):
            key = doc.text[:80]
            scores[key] = scores.get(key,0) + self.dw/(k_rrf+rank+1)
            dmap[key]   = doc
        for rank, doc in enumerate(sparse):
            key = doc.text[:80]
            scores[key] = scores.get(key,0) + self.sw/(k_rrf+rank+1)
            if key not in dmap: dmap[key] = doc
            else: dmap[key].sparse_score = doc.sparse_score
        merged = []
        for key, sc in sorted(scores.items(), key=lambda x:-x[1]):
            d = dmap[key]; d.final_score = round(sc,5); merged.append(d)
        return merged

    # ── Cross-encoder re-rank ─────────────────────────────────
    def _rerank(self, query: str, candidates: list, top_n: int) -> list:
        if not candidates: return []
        scores = self.reranker.predict([(query, d.text) for d in candidates])
        for d, s in zip(candidates, scores):
            d.rerank_score = round(float(s),4)
            boost = self.PRIORITY_BOOST.get(d.metadata.get('source_priority','low'),0.0)
            d.final_score  = round(d.rerank_score + boost, 4)
        candidates.sort(key=lambda x:-x.final_score)
        return candidates[:top_n]

    # ── Public retrieve ───────────────────────────────────────
    def retrieve(self, query: str, k: int = 5, fetch_k: int = 20,
                 use_reranker: bool = True,
                 source_filter: Optional[str] = None) -> list:
        # Route query to appropriate FAISS index via FAISSRetriever
        if source_filter:
            self.fr.retrieve.__func__  # ensure correct routing
        dense  = self._dense_retrieve(query, fetch_k)
        sparse = self._sparse_retrieve(query, fetch_k)
        if source_filter:
            dense  = [d for d in dense  if d.source == source_filter]
            sparse = [d for d in sparse if d.source == source_filter]
        merged = self._rrf(dense, sparse)
        return self._rerank(query, merged[:fetch_k], top_n=k) if use_reranker else merged[:k]


hybrid_retriever = HybridRetriever(
    faiss_retriever = faiss_retriever,
    bm25_index      = bm25_index,
    chunked_docs    = chunked_docs,
    reranker        = reranker,
)

print("✓ HybridRetriever ready (FAISS + BM25 + RRF + cross-encoder)")
test = hybrid_retriever.retrieve('broken product want refund', k=3)
for r in test:
    print(f"  [{r.source}] dense={r.dense_score:.3f} rerank={r.rerank_score:.3f}  {r.text[:80]}...")


✓ HybridRetriever ready (FAISS + BM25 + RRF + cross-encoder)
  [olist_review] dense=0.000 rerank=-0.269  Order ID: 216c6c60eaa02facee10abb4ca3a9c6f. Category: computers_accessories. Rev...
  [olist_review] dense=0.000 rerank=-0.628  . So I canceled the order and want a refund....
  [policy] dense=0.408 rerank=-0.668  1.4 REFUND ELIGIBILITY CONDITIONS
Full refund is issued when:
- Product is retur...


## LLM Setup (Groq) 

In [33]:
# PHASE 3 — CELL 5 — UNCHANGED
# Copy this exactly from the original Phase 3 guide
!pip install -q langchain-groq==0.1.3
from langchain_groq import ChatGroq
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ['GROQ_API_KEY'] = secrets.get_secret('GROQ_API_KEY')

llm = ChatGroq(model='llama-3.1-8b-instant', temperature=0.1, max_tokens=600)

test = llm.invoke('Say: ready')
print(f"✓ LLM ready: {test.content}")


✓ LLM ready: Go.


## Intent Classifier 

In [34]:
# Cell 6 — Rule-based intent classifier
# Rule-based first — fast, interpretable, no LLM cost
# LLM-based fallback only for ambiguous queries

from enum import Enum

class QueryIntent(str, Enum):
    ORDER_STATUS   = 'order_status'
    REFUND_REQUEST = 'refund_request'
    PRODUCT_ISSUE  = 'product_issue'
    DELIVERY_ISSUE = 'delivery_issue'
    POLICY_QUERY   = 'policy_query'
    SELLER_ISSUE   = 'seller_issue'
    GENERAL        = 'general'

INTENT_RULES = {
    QueryIntent.REFUND_REQUEST: [
        'refund','money back','reimbur','cancel','chargeback','return'],
    QueryIntent.ORDER_STATUS: [
        'where is my order','order status','tracking','track','shipped','dispatch'],
    QueryIntent.DELIVERY_ISSUE: [
        'late','delayed','not arrived','overdue','estimated date','never came'],
    QueryIntent.PRODUCT_ISSUE: [
        'broken','defective','damage','wrong item','not as described',
        'different','fake','counterfeit','missing part','not working'],
    QueryIntent.POLICY_QUERY: [
        'policy','how long','can i return','am i eligible','rules',
        'rights','consumer','CDC','within how many days'],
    QueryIntent.SELLER_ISSUE: [
        'seller','vendor','merchant','not responding','no reply'],
}

def classify_intent(query: str) -> QueryIntent:
    q = query.lower()
    for intent, keywords in INTENT_RULES.items():
        if any(kw in q for kw in keywords):
            return intent
    return QueryIntent.GENERAL

# Test
test_queries = [
    'My order has not arrived, it is 5 days late',
    'I want a refund for my broken laptop',
    'What is the return policy for electronics?',
]
for q in test_queries:
    print(f'  [{classify_intent(q).value:20s}] {q}')


  [delivery_issue      ] My order has not arrived, it is 5 days late
  [refund_request      ] I want a refund for my broken laptop
  [refund_request      ] What is the return policy for electronics?


## format_context — Fixed Typing Import + Safe Attribute Access

In [35]:
# PHASE 3 — CELL 7 — CHANGED: format_context with safe attribute access
from typing import List  # FIXED: was "from typing import list as List" (invalid)

def format_context(docs, max_chars: int = 2000, add_scores: bool = False) -> str:
    """
    Formats retrieved documents into a numbered context block.
    Works with both RetrievedDoc objects and plain dicts.
    """
    SOURCE_LABELS = {
        'policy':          'PLATFORM POLICY',
        'olist_review':    'CUSTOMER REVIEW',
        'support_ticket':  'SUPPORT RECORD',
        'amazon_complaint':'PRODUCT FEEDBACK',
        'seller_kpi':      'SELLER DATA',
    }
    sections, total = [], 0
    for i, doc in enumerate(docs, 1):
        # Safe attribute access — works for RetrievedDoc and dict
        if hasattr(doc, 'text'):
            text   = doc.text
            source = doc.source
            score  = getattr(doc, 'rerank_score', getattr(doc, 'dense_score', 0.0))
        else:
            text   = doc.get('text', '')
            source = doc.get('source', '')
            score  = doc.get('similarity', doc.get('rerank_score', 0.0))

        label     = SOURCE_LABELS.get(source, source.upper().replace(chr(95), chr(32)))
        score_str = f' [relevance: {score:.2f}]' if add_scores else ''
        header    = f'[Source {i} — {label}]{score_str}'
        body      = text.strip()
        remaining = max_chars - total - len(header) - 4
        if remaining <= 50: break
        if len(body) > remaining: body = body[:remaining] + '...'
        section = f'{header}\n{body}'
        sections.append(section)
        total += len(section)
    return '\n\n'.join(sections)

# Test
test_docs = hybrid_retriever.retrieve('return policy for damaged goods', k=3)
ctx = format_context(test_docs)
print(ctx[:400])


[Source 1 — PLATFORM POLICY]
1.2 EXTENDED RETURN FOR DEFECTIVE PRODUCTS
For defective or damaged products, the return window is extended to 30 days
for non-durable goods and 90 days for durable goods (electronics, appliances,
furniture), per CDC Article 26.

1.3 REFUND PROCESSING TIMELINE
- Credit card payments: 5-10 business days (up to 2 billing cycles)
- Boleto bancario: 3-5 business days to th


## Prompt Library

In [37]:
# Cell 8 — Complete prompt template library
# from langchain.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate

# ── Base system instruction shared by all templates ───────────
BASE_RULES = '''
STRICT RULES:
1. Answer ONLY using the retrieved sources provided below.
2. Cite source numbers in parentheses: (Source 1), (Source 2).
3. If the sources do not contain enough information, respond with:
   'I don't have enough information in my records to answer this precisely.
    Let me escalate this to a specialist who can help you further.'
4. Never invent order IDs, dates, prices, or policy details.
5. Be empathetic: acknowledge the customer's frustration before solving.
6. End every response with a clear next-step action the customer can take.
'''

# ── ORDER STATUS ──────────────────────────────────────────────
ORDER_STATUS_PROMPT = PromptTemplate(
    input_variables=['context', 'question', 'order_id'],
    template=f'''You are an expert Olist customer support agent
specialising in order tracking and delivery management.
{BASE_RULES}
RETRIEVED SOURCES:
{{context}}

CUSTOMER QUESTION: {{question}}
ORDER ID: {{order_id}}

Your response should:
- State the current delivery status clearly
- If delayed: acknowledge the delay, state how many days, cite policy
- Provide the estimated resolution timeline
- Give one concrete next step

RESPONSE:'''
)

# ── REFUND REQUEST ────────────────────────────────────────────
REFUND_PROMPT = PromptTemplate(
    input_variables=['context', 'question', 'order_id', 'days_since_purchase'],
    template=f'''You are an Olist refund and returns specialist.
{BASE_RULES}
RETRIEVED SOURCES:
{{context}}

CUSTOMER QUESTION: {{question}}
ORDER ID: {{order_id}}
DAYS SINCE PURCHASE: {{days_since_purchase}}

Your response should:
- State clearly whether the customer is eligible for a refund (based on sources)
- Cite the specific policy clause (Source number)
- Give exact processing timeframe in business days
- Provide step-by-step return initiation instructions

RESPONSE:'''
)

# ── PRODUCT / DEFECT ISSUE ────────────────────────────────────
PRODUCT_ISSUE_PROMPT = PromptTemplate(
    input_variables=['context', 'question', 'order_id'],
    template=f'''You are an Olist product quality specialist.
{BASE_RULES}
RETRIEVED SOURCES:
{{context}}

CUSTOMER QUESTION: {{question}}
ORDER ID: {{order_id}}

Your response should:
- Acknowledge the defect/issue empathetically
- Reference any similar complaints found in sources
- State replacement or refund options per policy (cite source)
- Give clear instructions for initiating the resolution

RESPONSE:'''
)

# ── GENERAL / POLICY QUERY ────────────────────────────────────
GENERAL_PROMPT = PromptTemplate(
    input_variables=['context', 'question'],
    template=f'''You are a knowledgeable Olist customer support agent.
{BASE_RULES}
RETRIEVED SOURCES:
{{context}}

CUSTOMER QUESTION: {{question}}

RESPONSE:'''
)

# ── Template registry ─────────────────────────────────────────
PROMPT_REGISTRY = {
    QueryIntent.ORDER_STATUS:   ORDER_STATUS_PROMPT,
    QueryIntent.REFUND_REQUEST: REFUND_PROMPT,
    QueryIntent.PRODUCT_ISSUE:  PRODUCT_ISSUE_PROMPT,
    QueryIntent.DELIVERY_ISSUE: ORDER_STATUS_PROMPT,  # reuse order status
    QueryIntent.POLICY_QUERY:   GENERAL_PROMPT,
    QueryIntent.SELLER_ISSUE:   GENERAL_PROMPT,
    QueryIntent.GENERAL:        GENERAL_PROMPT,
}
print(f'✓ {len(PROMPT_REGISTRY)} prompt templates registered')


✓ 7 prompt templates registered


## Query Rewriting — Expansion Before Retrieval

In [38]:
# Cell 9 — Query rewriter using the LLM
# Customer queries are often short, vague, or contain spelling errors.
# Rewriting them before retrieval dramatically improves recall.

# from langchain.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

REWRITE_PROMPT = PromptTemplate(
    input_variables=['query'],
    template='''You are a query rewriting assistant for an e-commerce support system.
Your task: rewrite the customer query to be more specific and retrieval-friendly.

Rules:
- Expand abbreviations (e.g. 'ord' → 'order')
- Add relevant context terms (e.g. 'broken' → 'broken defective product quality')
- Keep it concise (max 2 sentences)
- Preserve the original intent completely
- Do NOT answer the question — only rewrite it

Original query: {query}
Rewritten query:'''
)

rewrite_chain = REWRITE_PROMPT | llm | StrOutputParser()

def rewrite_query(query: str) -> str:
    """Rewrite query for better retrieval. Falls back to original on error."""
    try:
        rewritten = rewrite_chain.invoke({'query': query})
        return rewritten.strip()
    except Exception:
        return query  # graceful fallback

# Test
test_cases = [
    'my ord is late pls help',
    'product brokn',
    'refund??',
    'delivery problem order abc123',
]
for q in test_cases:
    rw = rewrite_query(q)
    print(f'  ORIGINAL: {q}')
    print(f'  REWRITTEN: {rw}\n')


  ORIGINAL: my ord is late pls help
  REWRITTEN: My order is late, and I need assistance with the delivery status.

  ORIGINAL: product brokn
  REWRITTEN: Rewritten query: I'm experiencing issues with a broken defective product quality, can you assist me with a replacement or refund for the affected order?

  ORIGINAL: refund??
  REWRITTEN: Refund for a defective product or order.

  ORIGINAL: delivery problem order abc123
  REWRITTEN: Delivery issue with order ABC123.



## Multi-Query Retrieval 

In [39]:
# PHASE 3 — CELL 10 — CHANGED: multi_query_retrieve uses RetrievedDoc
# from langchain.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

MULTI_QUERY_PROMPT = PromptTemplate(
    input_variables=['question'],
    template='''Generate 3 different search queries to retrieve documents
that would help answer this customer support question.
Use different phrasings and synonyms. Output ONLY the 3 queries,
one per line, no numbering, no explanation.
Question: {question}
Queries:'''
)
multi_query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()

def multi_query_retrieve(question: str, retriever,
                          k_per_query: int = 4, final_k: int = 6) -> list:
    """Generate 3 sub-queries, retrieve for each, deduplicate, re-rank."""
    raw = multi_query_chain.invoke({'question': question})
    sub_queries = [q.strip() for q in raw.strip().split('\n') if q.strip()][:3]
    sub_queries.append(question)

    all_docs: dict = {}
    for sq in sub_queries:
        docs = retriever.retrieve(sq, k=k_per_query, use_reranker=False)
        for doc in docs:
            key = doc.text[:100]  # FIXED: doc.text not doc["text"]
            if key not in all_docs or doc.dense_score > all_docs[key].dense_score:
                all_docs[key] = doc

    candidates = list(all_docs.values())
    if len(candidates) > 1:
        scores = reranker.predict([(question, d.text) for d in candidates])
        for d, s in zip(candidates, scores):
            d.rerank_score = round(float(s), 4)
        candidates.sort(key=lambda x: -x.rerank_score)

    return candidates[:final_k]

docs = multi_query_retrieve('package lost seller not responding', hybrid_retriever)
print(f"Multi-query: {len(docs)} docs retrieved")
for d in docs[:3]:
    print(f"  [{d.source}] rerank={d.rerank_score:.3f}  {d.text[:80]}...")


Multi-query: 6 docs retrieved
  [support_ticket] rerank=2.703  . Resolved by: We are sorry your package appears to be lost. We have initiated a...
  [support_ticket] rerank=2.560  Support ticket (package lost). Priority: critical. Channel: email. Customer quer...
  [support_ticket] rerank=2.542  Support ticket (package lost). Priority: critical. Channel: email. Customer quer...


## Multi-Hop Retrieval — Iterative Query Refinement

In [40]:
# PHASE 3 — CELL 10A — NEW: Multi-hop retrieval
def multi_hop_retrieve(query: str, retriever,
                        hops: int = 2, k_per_hop: int = 3) -> list:
    """
    Iterative retrieval with LLM-guided query refinement.
    Hop 1: retrieve on original query.
    Hop 2: LLM reads context, generates refined query, retrieve again.
    Final: dedup + rerank union of all hops.
    """
    current_query = query
    all_docs: dict = {}

    for hop in range(hops):
        docs = retriever.retrieve(current_query, k=k_per_hop, use_reranker=False)
        for d in docs:
            key = d.text[:100]
            if key not in all_docs or d.dense_score > all_docs[key].dense_score:
                all_docs[key] = d

        if hop == hops - 1:
            break  # no refinement after last hop

        # Refine query from retrieved context
        context_snippet = " ".join([d.text[:150] for d in docs])
        refinement_prompt = (
            f"You are refining a customer support search query.\n"
            f"Original query: {query}\n"
            f"Context found: {context_snippet[:400]}\n"
            f"Write ONE focused follow-up search query to find missing information.\n"
            f"Output ONLY the query, nothing else."
        )
        try:
            refined = llm.invoke(refinement_prompt).content.strip()
            if refined and len(refined) < 200:
                current_query = refined
        except Exception:
            break  # use what we have

    # Final dedup + rerank
    candidates = list(all_docs.values())
    if len(candidates) > 1:
        scores = reranker.predict([(query, d.text) for d in candidates])
        for d, s in zip(candidates, scores):
            d.rerank_score = round(float(s), 4)
        candidates.sort(key=lambda x: -x.rerank_score)

    return candidates[:6]


# Test
hop_docs = multi_hop_retrieve("seller not responding and package never arrived",
                               hybrid_retriever, hops=2)
print(f"Multi-hop: {len(hop_docs)} docs from 2 hops")
for d in hop_docs[:3]:
    print(f"  [{d.source}] rerank={d.rerank_score:.3f}  {d.text[:80]}...")


Multi-hop: 4 docs from 2 hops
  [amazon_complaint] rerank=2.103  Product complaint — Electronics. Rating: 1.0/5. Issue type: general_complaint. C...
  [support_ticket] rerank=0.972  Support ticket (package lost). Priority: critical. Channel: email. Customer quer...
  [olist_review] rerank=-6.141  Order ID: 23847650c6dde17cb17a2b180af209bd. Category: telephony. Review score: 1...


## Context Compression — Remove Irrelevant Sentences

In [41]:
# Cell 11 — LLM-based context compression
# Retrieved chunks often contain irrelevant sentences.
# Compression keeps only the relevant parts, reducing noise for the generator.

COMPRESS_PROMPT = PromptTemplate(
    input_variables=['question', 'document'],
    template='''Given the customer support question below,
extract ONLY the sentences from the document that are directly relevant to answering it.
If no sentences are relevant, respond with: NO_RELEVANT_CONTENT
Do not add any new information or commentary.

Question: {question}
Document: {document}
Relevant extract:'''
)

compress_chain = COMPRESS_PROMPT | llm | StrOutputParser()

def compress_context(
    query:    str,
    docs:     list[RetrievedDoc],
    min_score: float = 0.3
) -> list[RetrievedDoc]:
    """
    Compress each retrieved doc to only relevant sentences.
    Drops docs where nothing is relevant.
    Trade-off: adds 1 LLM call per doc — use only when accuracy > speed.
    """
    compressed = []
    for doc in docs:
        if doc.rerank_score < min_score:
            continue  # skip low-confidence docs
        try:
            extract = compress_chain.invoke({
                'question': query,
                'document': doc.text
            })
            if 'NO_RELEVANT_CONTENT' not in extract and len(extract.strip()) > 20:
                doc.text = extract.strip()  # replace with compressed version
                compressed.append(doc)
        except Exception:
            compressed.append(doc)  # keep original on error
    return compressed

# Note: use compression selectively — it adds latency
# Recommended: use for POLICY queries only (high accuracy needed)
# Skip for ORDER_STATUS queries (speed matters more)
print('✓ Context compression function ready')


✓ Context compression function ready


# SECTION 6 — LANGGRAPH AGENT DESIGN

## Install LangGraph

In [43]:
# Cell 12 — Install LangGraph

# -------- In case of error uninstall and reinstall with versions mentioned dependencies --------

# !pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters langgraph langgraph-checkpoint langgraph-prebuilt

!pip install -q \
langchain==0.1.20 \
langchain-core==0.1.53 \
langchain-community==0.0.38 \
langgraph==0.0.40 \
langsmith==0.1.147

# # !pip install -q langgraph langsmith
# # !pip install -q langgraph==0.0.40 langsmith==0.1.147

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, Sequence
import operator

print('✓ LangGraph ready')


ImportError: cannot import name 'CheckpointAt' from 'langgraph.checkpoint.base' (/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/base/__init__.py)

## Tool Definitions

In [ ]:
# Cell 13 — Tool definitions
# Tools are typed functions the LLM can call via function-calling.
# Each tool has a clear docstring — this is what the LLM reads to decide when to use it.

import pandas as pd
from langchain_core.tools import tool

# Load order data once at startup (fast pandas lookup)
order_df = pd.read_parquet('/kaggle/input/notebooks/yashasdevang/preprocessing/processed/order_lookup.parquet')
seller_df = pd.read_parquet('/kaggle/input/notebooks/yashasdevang/preprocessing/processed/seller_kpi.parquet')


@tool
def rag_search(query: str) -> str:
    """
    Search the knowledge base for policies, product complaints, and support history.
    Use this for: return policy, refund rules, product quality issues, similar complaints.
    Input: a natural language search query.
    """
    docs = hybrid_retriever.retrieve(query, k=4)
    return format_context(docs, max_chars=1200)


@tool
def order_lookup(order_id: str) -> str:
    """
    Look up order status, delivery dates, and payment information by order ID.
    Use this when the customer provides an order ID or asks about a specific order.
    Input: the order ID string (e.g. 'abc123def456').
    """
    row = order_df[order_df['order_id'] == order_id]
    if row.empty:
        return f'No order found with ID: {order_id}. Please verify the order ID.'
    r = row.iloc[0]
    delay = r.get('delivery_delay_days', None)
    delay_str = f'{int(delay)} days late' if pd.notna(delay) and delay > 0 else 'on time'
    return (
        f'Order {order_id}: status={r.order_status}, '
        f'purchased={r.order_purchase_timestamp}, '
        f'estimated_delivery={r.order_estimated_delivery_date}, '
        f'actual_delivery={r.order_delivered_customer_date}, '
        f'delivery={delay_str}, '
        f'payment_type={r.payment_type}, '
        f'payment_value=R${r.payment_value:.2f}, '
        f'category={r.category_en}, '
        f'issue_flag={r.issue_category}'
    )


@tool
def seller_analysis(seller_id: str) -> str:
    """
    Get seller performance metrics including complaint rate, late delivery rate, and rating.
    Use this when a customer reports issues with a specific seller.
    Input: the seller ID string.
    """
    row = seller_df[seller_df['seller_id'] == seller_id]
    if row.empty:
        return f'No seller data for ID: {seller_id}'
    r = row.iloc[0]
    return (
        f'Seller {seller_id}: avg_rating={r.avg_rating:.1f}/5, '
        f'late_rate={r.late_rate*100:.1f}%, '
        f'complaint_rate={r.complaint_rate*100:.1f}%, '
        f'total_orders={int(r.total_orders)}, '
        f'flagged={r.is_flagged}'
    )


@tool
def escalate_to_human(reason: str) -> str:
    """
    Escalate the conversation to a human support agent.
    Use this when: the issue is complex, customer is very upset, fraud is suspected,
    or when 2 tool calls have not resolved the issue.
    Input: a brief description of why escalation is needed.
    """
    return (
        f'ESCALATED: This case has been flagged for human review. '
        f'Reason: {reason}. '
        f'A support specialist will contact you within 4 hours via email.'
    )


TOOLS = [rag_search, order_lookup, seller_analysis, escalate_to_human]
print(f'✓ {len(TOOLS)} tools defined')


## Agent State — Memory and Context

In [ ]:
# Cell 14 — Agent state definition
# The state is the shared memory that flows through every node in the graph.
# Using Annotated[list, operator.add] means messages are APPENDED not replaced.

from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
import operator

class AgentState(TypedDict):
    # Conversation history — accumulates across turns
    messages:         Annotated[Sequence[BaseMessage], operator.add]
    # Extracted from query if present
    order_id:         str
    # Classified intent for routing decisions
    intent:           str
    # Number of tool calls made this turn (prevents infinite loops)
    tool_call_count:  int
    # Flag set by escalation tool
    escalated:        bool
    # Final answer accumulated
    final_answer:     str

print('✓ AgentState defined')


## Graph Nodes — Agent Logic

In [ ]:
# Cell 15 — Graph node definitions

from langchain_core.messages import SystemMessage

# Bind tools to the LLM (enables function-calling)
llm_with_tools = llm.bind_tools(TOOLS)

SYSTEM_PROMPT = '''You are an expert Olist e-commerce customer support agent.
You have access to tools to look up orders, search knowledge base, and analyse sellers.

DECISION FRAMEWORK:
1. If the customer mentions an order ID → call order_lookup FIRST
2. If the question involves policy/refunds/returns → call rag_search
3. If seller is mentioned → call seller_analysis
4. If you have all information needed → generate final response WITHOUT tool calls
5. If issue cannot be resolved after 2 tool calls → call escalate_to_human

Always be empathetic, cite your sources, and give a concrete next step.'''


def agent_node(state: AgentState) -> dict:
    """Main reasoning node — decides what tools to call or generates final answer"""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + list(state['messages'])
    response = llm_with_tools.invoke(messages)
    return {
        'messages': [response],
        'tool_call_count': state.get('tool_call_count', 0)
    }


def extract_metadata_node(state: AgentState) -> dict:
    """Extract order ID and intent from the latest user message"""
    last_msg = state['messages'][-1].content if state['messages'] else ''

    # Extract order ID (simple regex — works for Olist UUID-like IDs)
    import re
    order_match = re.search(r'\b([a-f0-9]{32})\b', last_msg)
    order_id    = order_match.group(1) if order_match else ''

    intent = classify_intent(last_msg).value

    return {'order_id': order_id, 'intent': intent}


def tools_node(state: AgentState) -> dict:
    """Executes tool calls, increments counter, checks escalation"""
    tool_executor = ToolNode(TOOLS)
    result = tool_executor.invoke(state)
    count  = state.get('tool_call_count', 0) + 1
    # Check if escalation was triggered
    last_msg = result.get('messages', [{}])[-1]
    escalated = 'ESCALATED' in str(getattr(last_msg, 'content', ''))
    return {**result, 'tool_call_count': count, 'escalated': escalated}


## Routing Logic — Conditional Edges

In [ ]:
# Cell 16 — Routing functions (conditional edges)

def should_continue(state: AgentState) -> str:
    """
    After the agent node, decide:
    - 'tools'    → agent wants to call a tool
    - 'end'      → agent is done, return answer
    - 'escalate' → max tool calls reached
    """
    last_msg = state['messages'][-1]
    count    = state.get('tool_call_count', 0)

    # Guard: max 3 tool calls per query to prevent infinite loops
    if count >= 3:
        return 'end'

    # If escalated, stop
    if state.get('escalated', False):
        return 'end'

    # Check if the LLM made tool calls
    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
        return 'tools'

    return 'end'


def after_tools(state: AgentState) -> str:
    """After tools execute, always return to agent for synthesis"""
    if state.get('escalated', False):
        return 'end'
    return 'agent'


## Compile the Full LangGraph

In [ ]:
# Cell 17 — Build and compile the LangGraph
from langgraph.graph import StateGraph, END

# ── Build graph ───────────────────────────────────────────────
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node('extract_metadata', extract_metadata_node)
workflow.add_node('agent',            agent_node)
workflow.add_node('tools',            tools_node)

# Entry point
workflow.set_entry_point('extract_metadata')

# Static edges
workflow.add_edge('extract_metadata', 'agent')

# Conditional edges — routing logic
workflow.add_conditional_edges(
    'agent',
    should_continue,
    {
        'tools': 'tools',
        'end':   END,
    }
)

workflow.add_conditional_edges(
    'tools',
    after_tools,
    {
        'agent': 'agent',
        'end':   END,
    }
)

# Compile
agent_app = workflow.compile()

print('✓ LangGraph agent compiled')
print('Graph nodes:', list(workflow.nodes.keys()))


## ConversationMemory

In [ ]:
# PHASE 3 — CELL 18A — NEW: Conversation Memory
# Run this BEFORE Cell 18 (OlistRAGSystem)
from collections import defaultdict

class ConversationMemory:
    """
    Per-session conversation history with sliding context window.
    Production upgrade: replace self.store with Redis or a database.
    """
    def __init__(self, max_turns: int = 5):
        self.store     = defaultdict(list)
        self.max_turns = max_turns

    def add_turn(self, session_id: str, user_query: str, response: str):
        """Add one Q&A turn. Keep only most recent max_turns."""
        self.store[session_id].append({'user': user_query, 'assistant': response})
        if len(self.store[session_id]) > self.max_turns:
            self.store[session_id] = self.store[session_id][-self.max_turns:]

    def get_history(self, session_id: str) -> str:
        """Return formatted history string for context injection."""
        turns = self.store.get(session_id, [])
        if not turns: return ''
        lines = []
        for t in turns:
            lines.append(f"User: {t['user']}")
            lines.append(f"Assistant: {t['assistant'][:200]}...")  # truncate saves tokens
        return "\n".join(lines)

    def clear(self, session_id: str):
        self.store.pop(session_id, None)

    def sessions(self) -> list:
        return list(self.store.keys())


memory_store = ConversationMemory(max_turns=5)
print("✓ ConversationMemory ready")

# Quick test
memory_store.add_turn('test', 'What is return policy?', 'Returns within 7 days.')
memory_store.add_turn('test', 'What about electronics?', 'Electronics: 30-day warranty.')
print("Test history:")
print(memory_store.get_history('test'))
memory_store.clear('test')
print("✓ Memory test passed")


## OlistRAGSystem — Updated with Feedback Loop + Memory

In [ ]:
# PHASE 3 — CELL 18 — CHANGED: OlistRAGSystem with memory + feedback loop
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from typing import Optional
import time

LOW_CONFIDENCE = [
    "don't have enough information",
    "not enough information",
    "cannot find",
    "i need to check this further",
    "i don't have",
]


class OlistRAGSystem:
    """
    Production RAG + Agent system.
    Fast path: rewrite → retrieve → compress → generate
    Slow path: LangGraph agent with tools
    NEW: conversation memory + feedback loop retry
    """
    AGENT_TRIGGERS = ['order_status', 'delivery_issue', 'seller_issue']

    def __init__(self, retriever, llm, agent_app,
                 use_rewrite=True, use_multiquery=False,
                 use_compress=False, use_multihop=False):
        self.retriever   = retriever
        self.llm         = llm
        self.agent       = agent_app
        self.use_rewrite = use_rewrite
        self.use_mq      = use_multiquery
        self.use_comp    = use_compress
        self.use_mh      = use_multihop

    def _should_use_agent(self, query: str) -> bool:
        return classify_intent(query).value in self.AGENT_TRIGGERS

    def _rag_chain_answer(self, query: str, order_id: str = '',
                          session_id: str = 'default') -> dict:
        t0 = time.time()
        original_query = query  # IMPORTANT: keep original before any expansion

        # ── Step 1: Inject conversation history ───────────────
        # History is injected into RETRIEVAL query only
        # original_query is saved to memory (NOT the expanded version)
        history = memory_store.get_history(session_id)
        search_query = (
            f"Previous context:\n{history}\n\nCurrent question: {query}"
            if history else query
        )

        # ── Step 2: Query rewriting ───────────────────────────
        if self.use_rewrite:
            search_query = rewrite_query(search_query)

        # ── Step 3: Retrieval ─────────────────────────────────
        if self.use_mh:
            docs = multi_hop_retrieve(search_query, self.retriever)
        elif self.use_mq:
            docs = multi_query_retrieve(search_query, self.retriever)
        else:
            docs = self.retriever.retrieve(search_query, k=5)

        # ── Step 4: Optional compression ─────────────────────
        if self.use_comp:
            if classify_intent(query) == QueryIntent.POLICY_QUERY:
                docs = compress_context(query, docs)

        # ── Step 5: Format context ────────────────────────────
        context = format_context(docs, max_chars=2000)

        # ── Step 6: Generate answer ───────────────────────────
        intent = classify_intent(query)
        prompt = PROMPT_REGISTRY[intent]
        chain  = prompt | self.llm | StrOutputParser()
        kwargs = {'context': context, 'question': query}
        if 'order_id' in prompt.input_variables:
            kwargs['order_id'] = order_id or 'not provided'
        if 'days_since_purchase' in prompt.input_variables:
            kwargs['days_since_purchase'] = 'unknown'
        answer = chain.invoke(kwargs)

        # ── Step 7: Feedback loop ─────────────────────────────
        if any(p in answer.lower() for p in LOW_CONFIDENCE):
            print("  ↺ Low confidence → retrying with multi-query")
            retry_docs = multi_query_retrieve(original_query, self.retriever)
            kwargs['context'] = format_context(retry_docs, max_chars=2000)
            answer = chain.invoke(kwargs)
            docs   = retry_docs

        # ── Step 8: Save to memory (original query only) ──────
        memory_store.add_turn(session_id, original_query, answer)

        return {
            'answer':     answer,
            'sources':    [d.source for d in docs],
            'intent':     intent.value,
            'mode':       'rag_chain',
            'session_id': session_id,
            'latency_ms': round((time.time()-t0)*1000, 1),
            'docs_used':  len(docs),
        }

    def _agent_answer(self, query: str, session_id: str = 'default') -> dict:
        t0 = time.time()
        history = memory_store.get_history(session_id)
        full_q  = f"Context:\n{history}\n\nQuery: {query}" if history else query
        result  = self.agent.invoke({
            'messages': [HumanMessage(content=full_q)],
            'order_id': '', 'intent': '',
            'tool_call_count': 0, 'escalated': False, 'final_answer': ''
        })
        last_msg = result['messages'][-1]
        answer   = getattr(last_msg, 'content', str(last_msg))
        memory_store.add_turn(session_id, query, answer)
        return {
            'answer':     answer,
            'intent':     result.get('intent',''),
            'mode':       'agent',
            'session_id': session_id,
            'latency_ms': round((time.time()-t0)*1000, 1),
            'tool_calls': result.get('tool_call_count', 0),
            'escalated':  result.get('escalated', False),
        }

    def answer(self, query: str, order_id: str = '',
               session_id: str = 'default') -> dict:
        if self._should_use_agent(query):
            return self._agent_answer(query, session_id)
        return self._rag_chain_answer(query, order_id, session_id)


rag_system = OlistRAGSystem(
    retriever=hybrid_retriever, llm=llm, agent_app=agent_app,
    use_rewrite=True, use_multiquery=False,
    use_compress=False, use_multihop=False,
)
print("✓ OlistRAGSystem ready")
print("  Memory: ConversationMemory (5-turn sliding window)")
print("  Feedback loop: auto-retry on low-confidence answers")


## End-to-End Test + Multi-Turn Memory Test

In [ ]:
# PHASE 3 — CELL 19 — CHANGED: E2E test with session IDs
TEST_QUERIES = [
    {'query': 'What is the return policy for damaged goods?',
     'session_id': 's1', 'expected_intent': 'policy_query', 'expected_mode': 'rag_chain'},
    {'query': 'My order is delayed by 10 days, what do I do?',
     'session_id': 's2', 'expected_intent': 'delivery_issue', 'expected_mode': 'agent'},
    {'query': 'I received a broken electronic product and need a refund',
     'session_id': 's3', 'expected_intent': 'product_issue', 'expected_mode': 'rag_chain'},
    {'query': 'How long does a refund take after returning an item?',
     'session_id': 's4', 'expected_intent': 'refund_request', 'expected_mode': 'rag_chain'},
    {'query': 'The seller has not responded in 7 days and my order never arrived',
     'session_id': 's5', 'expected_intent': 'seller_issue', 'expected_mode': 'agent'},
]

print("=== END-TO-END TEST ===")
for i, tc in enumerate(TEST_QUERIES, 1):
    result = rag_system.answer(tc['query'], session_id=tc['session_id'])
    oi = '✓' if result.get('intent') == tc['expected_intent'] else '✗'
    om = '✓' if result.get('mode')   == tc['expected_mode']   else '✗'
    print(f"[TC-{i:02d}] {oi}intent={result.get('intent')}  {om}mode={result.get('mode')}  {result.get('latency_ms')}ms")
    print(f"        {result['answer'][:150]}...")

# ── Multi-turn memory test ──────────────────────────────────────────────
print("\n=== MULTI-TURN MEMORY TEST ===")

r1 = rag_system.answer("What is the return window for electronics?", session_id="mem_test")
print(f"Turn 1: {r1['answer'][:150]}...")

r2 = rag_system.answer("And what if it arrived damaged?", session_id="mem_test")
print(f"Turn 2: {r2['answer'][:200]}...")
print("  ↑ Turn 2 should reference electronics/return context from Turn 1")

print(f"\nActive sessions: {memory_store.sessions()}")


In [ ]:
# EXPORT FOR EVALUATION NOTEBOOK

import pandas as pd

tickets = pd.read_parquet('/kaggle/input/notebooks/yashasdevang/preprocessing/processed/synthetic_tickets.parquet')
eval_sample = tickets.sample(10, random_state=42)

eval_data = []

for j, (_, row) in enumerate(eval_sample.iterrows()):
    print(f"Processing {j+1}/{len(eval_sample)}...")
    query = row['ticket_text']
    sid   = f'eval_{j}'

    result = rag_system.answer(query, session_id=sid)
    docs   = hybrid_retriever.retrieve(query, k=4)

    eval_data.append({
        "question": query,
        "answer": result['answer'],
        "contexts": [d.text for d in docs],
        "ground_truth": row['resolution_text']
    })

df_eval = pd.DataFrame(eval_data)

# Save
df_eval.to_json('/kaggle/working/eval_data.json', orient='records', indent=2)

print("✓ Evaluation data saved")

# ============== USER INTERFACE ==============

In [ ]:
# Add this cell AFTER all your Phase 3 cells in the Kaggle notebook
# (after rag_system, hybrid_retriever, memory_store are all defined)

# Cell XX — Launch Streamlit UI
!pip install -q streamlit

# Write app.py to disk
# Copy the entire app.py content into this cell using %%writefile
# OR upload app.py to Kaggle as a dataset and reference it

# Option A: inline writefile (paste app.py content after %%writefile)
# %%writefile app.py
# <paste entire app.py here>

# Option B: if app.py is uploaded as kaggle dataset
# import shutil
# shutil.copy("/kaggle/input/olist-ui/app.py", "app.py")

# Launch
from pyngrok import ngrok
!pip install -q pyngrok

# Set your ngrok authtoken (free at ngrok.com)
from kaggle_secrets import UserSecretsClient
import os
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))

# Start Streamlit in background
import subprocess, threading
proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Create public URL via ngrok
import time; time.sleep(3)
public_url = ngrok.connect(8501)
print(f"✓ App is live at: {public_url}")
print("  Open this URL in your browser")
